# TRP lower-gate survey

Diagonal lower-gate distances in mammalian homotetrameric TRP cryo-EM structures, split by S6 register (α or π). Output: `lower_gate_survey.csv`.

In [ ]:
import os, csv, json, re, urllib.request
import numpy as np
from collections import Counter, defaultdict

cif_dir = "cif"
one_letter = dict(ALA="A", ARG="R", ASN="N", ASP="D", CYS="C", GLN="Q", GLU="E", GLY="G", HIS="H", ILE="I",
                  LEU="L", LYS="K", MET="M", PHE="F", PRO="P", SER="S", THR="T", TRP="W", TYR="Y", VAL="V", MSE="M")
hydrophobic = {"ALA", "ILE", "LEU", "MET", "PHE", "VAL", "TRP", "CYS"}
backbone_atoms = {"N", "CA", "C", "O", "OXT"}
channel_pattern = re.compile(r"transient receptor|TRP|polycystin|polycystic|mucolipin|melastatin|nompc|mechanoreceptor potential|vanilloid|predicted protein|YVC1|TRPY", re.I)
mammals = {"Homo sapiens", "Mus musculus", "Rattus norvegicus", "Oryctolagus cuniculus", "ORYCTOLAGUS CUNICULUS",
           "Bos taurus", "Ictidomys tridecemlineatus", "Callithrix jacchus"}
fusion_tag_organisms = {"Escherichia coli", "Escherichia coli K-12", "Escherichia coli (strain K12)", "Aequorea victoria", "Human betaherpesvirus 5",
                 "Human cytomegalovirus", "synthetic construct", "Muromegalovirus G4", "Pseudotevenvirus RB43"}
fusion_tag_uniprot = {"P0AEX9", "P42212", "P00720", "P0ABE7", "P08318", "P0DP23", "P0DP24", "P0DP25"}
trpm8_uniprot = ("Q8R4D5", "Q7Z2W7")

# S6 around the pi lower gate (LG): three hydrophobic, the conserved Asn (Ser in TRPML), then L-L-LG-A-L
s6_motif = ["LIVMF", "LIVMF", "LIVMF", "NSQ", "MLF", "LF", "ILVF", "AG", "LIMVF"]
lg_index = 6
# alpha position relative to the pi-LG is -1, except mouse TRPV4 and TRPM8 (+3)
alpha_shift_exceptions = {"Q9EPK8": 3, "Q8R4D5": 3, "Q7Z2W7": 3}

## 1. Search RCSB PDB

One full-text search per channel name for EM entries released up to 2026-09-16. Entries are kept if the title or a polymer entity name matches a TRP keyword.

In [ ]:
search_terms = ["TRPV1", "TRPV2", "TRPV3", "TRPV4", "TRPV5", "TRPV6", "TRPC1", "TRPC3", "TRPC4", "TRPC5", "TRPC6", "TRPgamma",
                "TRPM1", "TRPM2", "TRPM3", "TRPM4", "TRPM5", "TRPM7", "TRPM8", "TRPML1", "TRPML2", "TRPML3", "TRPA1",
                "PKD1", "PKD2", "PKD1L3", "PKD2L1", "NOMPC", "TRPY1"]
hits = set()
for term in search_terms:
    query = {"query": {"type": "group", "logical_operator": "and", "nodes": [
                 {"type": "terminal", "service": "full_text", "parameters": {"value": term}},
                 {"type": "terminal", "service": "text", "parameters": {"attribute": "exptl.method", "operator": "exact_match", "value": "ELECTRON MICROSCOPY"}},
                 {"type": "terminal", "service": "text", "parameters": {"attribute": "rcsb_accession_info.initial_release_date", "operator": "less_or_equal", "value": "2026-09-16"}}]},
             "return_type": "entry", "request_options": {"return_all_hits": True}}
    request = urllib.request.Request("https://search.rcsb.org/rcsbsearch/v2/query", json.dumps(query).encode(), {"Content-Type": "application/json"})
    hits |= {hit["identifier"] for hit in json.load(urllib.request.urlopen(request))["result_set"]}

# title and polymer entities (name, UniProt, source organism) of each hit
entries = {}
pdb_ids = sorted(hits)
for start in range(0, len(pdb_ids), 200):
    graphql = ("{entries(entry_ids:%s){rcsb_id struct{title} polymer_entities{rcsb_polymer_entity_container_identifiers{entity_id uniprot_ids} "
               "rcsb_polymer_entity{pdbx_description} rcsb_entity_source_organism{scientific_name}}}}" % json.dumps(pdb_ids[start:start + 200]))
    request = urllib.request.Request("https://data.rcsb.org/graphql", json.dumps({"query": graphql}).encode(), {"Content-Type": "application/json"})
    for entry in json.load(urllib.request.urlopen(request))["data"]["entries"]:
        entities = {}
        for entity in entry["polymer_entities"] or []:
            identifiers = entity["rcsb_polymer_entity_container_identifiers"]
            entities[identifiers["entity_id"]] = {
                "name": entity["rcsb_polymer_entity"]["pdbx_description"] or "",
                "uniprot": identifiers["uniprot_ids"] or [],
                "organism": [source["scientific_name"] for source in entity["rcsb_entity_source_organism"] or []]}
        names = [entry["struct"]["title"] or ""] + [entity["name"] for entity in entities.values()]
        if any(channel_pattern.search(name) for name in names):
            entries[entry["rcsb_id"]] = entities
print(len(entries), "entries")

## 2. Download structures

In [ ]:
os.makedirs(cif_dir, exist_ok=True)
for pdb_id in entries:
    path = f"{cif_dir}/{pdb_id}.cif"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.cif", path)

## 3. Survey

Filters, in order: (1) not a tetramer, (2) no S6, (3) not mammalian, (4) heteromer, (6) chains disagree on α/π, (5) the two diagonal LG Cα–Cα distances differ by more than 2 Å.

The π-LG (X) is the S6 residue three after the conserved Asn (Ser in TRPML), found in each chain by the sequence pattern match closest to the pore axis. The S6 is π if the X side chain points more directly at the pore axis than the α position (X−1, or X+3 for mouse TRPV4 and TRPM8), otherwise α.

Fig 1 residue: X for π, X−1 for α (X+3 for TRPM8 α). Fig 2 residue: four further along. For each, the Cα–Cα and closest heavy-atom distances are averaged over the two diagonal chain pairs. The residue is not used if its Cα→Cβ points away from the pore axis by more than 90° or its inward reach (Cα distance from the axis minus that of the closest side-chain atom) is below 1 Å.

In [ ]:
def read_atoms(path):
    """Protein heavy atoms of the first model of an mmCIF file, as arrays."""
    columns = []
    records = []
    for line in open(path):
        if line.startswith("_atom_site."):
            columns.append(line.split(".", 1)[1].strip())
        elif columns and line.startswith(("ATOM", "HETATM")):
            records.append(line.split())
        elif records:
            break
    column = {name: index for index, name in enumerate(columns)}
    model = records[0][column["pdbx_PDB_model_num"]] if "pdbx_PDB_model_num" in column else None
    atoms = [record for record in records
             if (model is None or record[column["pdbx_PDB_model_num"]] == model)
             and record[column["type_symbol"]] not in ("H", "D")
             and record[column["label_comp_id"]] in one_letter
             and record[column["label_alt_id"]] in (".", "?", "A")]
    fields = ["auth_asym_id", "label_entity_id", "auth_seq_id", "label_comp_id", "label_atom_id"]
    arrays = [np.array([atom[column[field]] for atom in atoms]) for field in fields]
    coordinates = np.array([[float(atom[column[axis]]) for axis in ("Cartn_x", "Cartn_y", "Cartn_z")] for atom in atoms])
    return arrays + [coordinates]

In [ ]:
rows = []
for pdb_id in sorted(entries):
    row = {"pdb_id": pdb_id, "excluded": ""}
    rows.append(row)

    chain, entity, residue_number, residue_name, atom_name, coordinates = read_atoms(f"{cif_dir}/{pdb_id}.cif")
    is_ca = atom_name == "CA"

    # channel chains: channel entities with at least 150 CA
    channel_entities = [entity_id for entity_id, info in entries[pdb_id].items() if channel_pattern.search(info["name"])]
    chains = {}
    for chain_id in np.unique(chain[np.isin(entity, channel_entities)]):
        mask = (chain == chain_id) & is_ca
        if mask.sum() >= 150:
            chains[chain_id] = entity[mask][0]
    if len(chains) != 4:
        row["excluded"] = "1 not a tetramer"
        continue

    # pore axis: normal of the plane through the centroids of the identical chains
    main_entity = Counter(chains.values()).most_common(1)[0][0]
    identical_chains = [chain_id for chain_id, entity_id in chains.items() if entity_id == main_entity]
    common_residues = list(set.intersection(*[set(residue_number[(chain == chain_id) & is_ca]) for chain_id in identical_chains]))
    centroids = np.array([coordinates[(chain == chain_id) & is_ca & np.isin(residue_number, common_residues)].mean(0) for chain_id in identical_chains])
    if len(identical_chains) == 4:
        centre = centroids.mean(0)
        axis = np.linalg.svd(centroids - centre)[2][2]
    elif len(identical_chains) == 3:
        edge_1 = centroids[1] - centroids[0]
        edge_2 = centroids[2] - centroids[0]
        normal = np.cross(edge_1, edge_2)
        centre = centroids[0] + (edge_2 @ edge_2 * np.cross(normal, edge_1) + edge_1 @ edge_1 * np.cross(edge_2, normal)) / (2 * normal @ normal)
        axis = normal
    else:
        channel_ca_coordinates = coordinates[np.isin(chain, list(chains)) & is_ca]
        centre = channel_ca_coordinates.mean(0)
        eigenvalues, eigenvectors = np.linalg.eigh(np.cov((channel_ca_coordinates - centre).T))
        axis = eigenvectors[:, np.argmax([abs(eigenvalues[k] - np.mean(np.delete(eigenvalues, k))) for k in range(3)])]
    axis = axis / np.linalg.norm(axis)
    height = (coordinates - centre) @ axis
    radial = (coordinates - centre) - np.outer(height, axis)
    radius = np.linalg.norm(radial, axis=1)
    basis_1 = np.cross(axis, [1, 0, 0] if abs(axis[0]) < 0.9 else [0, 1, 0])
    basis_1 = basis_1 / np.linalg.norm(basis_1)
    basis_2 = np.cross(axis, basis_1)
    angle = np.arctan2(radial @ basis_2, radial @ basis_1)

    # transmembrane check: best hydrophobic fraction of the outer-surface CA in a 30 A slab
    channel_ca = is_ca & np.isin(chain, list(chains))
    slab = np.floor(height[channel_ca] / 2).astype(int)
    sector = ((angle[channel_ca] + np.pi) / (2 * np.pi) * 16).astype(int) % 16
    is_hydrophobic = np.isin(residue_name[channel_ca], list(hydrophobic))
    ca_radius = radius[channel_ca]
    slabs = np.arange(slab.min(), slab.max() + 1)
    hydrophobic_count = np.zeros(len(slabs))
    surface_count = np.zeros(len(slabs))
    for k, slab_index in enumerate(slabs):
        for sector_index in range(16):
            mask = (slab == slab_index) & (sector == sector_index)
            if mask.any():
                outer = mask & (ca_radius >= ca_radius[mask].max() - 5) & (ca_radius > 20)
                hydrophobic_count[k] += is_hydrophobic[outer].sum()
                surface_count[k] += outer.sum()
    membrane_score = max(hydrophobic_count[max(0, k - 7):k + 8].sum() / surface_count[max(0, k - 7):k + 8].sum()
                         if surface_count[max(0, k - 7):k + 8].sum() > 60 else 0 for k in range(len(slabs)))
    if membrane_score < 0.5:
        row["excluded"] = "2 no S6"
        continue

    # species and channel
    uniprot_ids = [accession for accession in entries[pdb_id][main_entity]["uniprot"] if accession not in fusion_tag_uniprot]
    organisms = [organism for organism in entries[pdb_id][main_entity]["organism"] if organism not in fusion_tag_organisms]
    organism = organisms[0] if organisms else "?"
    uniprot_id = uniprot_ids[0] if uniprot_ids else None
    row["protein"] = entries[pdb_id][main_entity]["name"]
    row["organism"] = organism
    row["uniprot"] = uniprot_id
    if organism not in mammals:
        row["excluded"] = "3 not mammalian"
        continue
    if len(set(chains.values())) > 1:
        row["excluded"] = "4 heteromer"
        continue

    # residues of the channel chains: (chain, residue number) -> atom name -> (coordinates, radial vector)
    residues = defaultdict(dict)
    sequence = defaultdict(dict)
    for index in np.where(np.isin(chain, list(chains)))[0]:
        try:
            number = int(residue_number[index])
        except ValueError:
            continue
        residues[(chain[index], number)][atom_name[index]] = (coordinates[index], radial[index])
        if atom_name[index] == "CA":
            sequence[chain[index]][number] = one_letter[residue_name[index]]

    # pi-LG in each chain: the S6 motif match closest to the pore axis
    lower_gate = {}
    for chain_id in chains:
        closest = None
        for number in sequence[chain_id]:
            if all(sequence[chain_id].get(number - lg_index + i, "-") in s6_motif[i] for i in range(len(s6_motif))):
                distance = np.linalg.norm(residues[(chain_id, number)]["CA"][1])
                if closest is None or distance < closest:
                    closest = distance
                    lower_gate[chain_id] = number
    if len(lower_gate) < 4:
        row["excluded"] = "2 no S6"
        continue

    # S6 register: cosine between CA -> side-chain centroid and CA -> pore axis, at X and at the alpha position
    alpha_shift = alpha_shift_exceptions.get(uniprot_id, -1)
    cosine = {}
    for chain_id in chains:
        for shift in (0, alpha_shift):
            residue_atoms = residues.get((chain_id, lower_gate[chain_id] + shift), {})
            side_chain = [vectors[1] for name, vectors in residue_atoms.items() if name not in backbone_atoms]
            if "CA" in residue_atoms and side_chain:
                to_side_chain = np.mean(side_chain, 0) - residue_atoms["CA"][1]
                to_axis = -residue_atoms["CA"][1]
                cosine[(chain_id, shift)] = to_side_chain @ to_axis / (np.linalg.norm(to_side_chain) * np.linalg.norm(to_axis))
    chain_registers = set()
    for chain_id in chains:
        cosine_pi = cosine.get((chain_id, 0))
        cosine_alpha = cosine.get((chain_id, alpha_shift))
        chain_registers.add("π" if cosine_alpha is None or (cosine_pi is not None and cosine_pi > cosine_alpha) else "α")
    mean_pi = np.mean([cosine[(chain_id, 0)] for chain_id in chains]) if all((chain_id, 0) in cosine for chain_id in chains) else None
    mean_alpha = np.mean([cosine[(chain_id, alpha_shift)] for chain_id in chains]) if all((chain_id, alpha_shift) in cosine for chain_id in chains) else None
    if mean_pi is None and mean_alpha is None:
        row["excluded"] = "2 no S6"
        continue
    register = "π" if mean_alpha is None or (mean_pi is not None and mean_pi > mean_alpha) else "α"
    row["S6"] = register

    # gate diagonals, chains ordered around the pore
    chain_angle = {}
    for chain_id in chains:
        offset = coordinates[(chain == chain_id) & is_ca].mean(0) - centre
        chain_angle[chain_id] = np.arctan2(offset @ basis_2, offset @ basis_1)
    ring = sorted(chains, key=chain_angle.get)
    diagonal_pairs = [(ring[0], ring[2]), (ring[1], ring[3])]
    gate_shift = 0 if register == "π" else alpha_shift
    diagonals = []
    for chain_1, chain_2 in diagonal_pairs:
        atoms_1 = residues.get((chain_1, lower_gate[chain_1] + gate_shift), {})
        atoms_2 = residues.get((chain_2, lower_gate[chain_2] + gate_shift), {})
        if "CA" in atoms_1 and "CA" in atoms_2:
            diagonals.append(round(float(np.linalg.norm(atoms_1["CA"][0] - atoms_2["CA"][0])), 2))
    if len(diagonals) < 2:
        row["excluded"] = "2 no S6"
        continue
    if len(chain_registers) > 1:
        row["excluded"] = "6 mixed alpha/pi"
        continue
    if abs(diagonals[0] - diagonals[1]) > 2.0:
        row["excluded"] = "5 asymmetric"
        continue

    # Fig 1 and Fig 2 residues
    fig1_shift = 0 if register == "π" else (3 if uniprot_id in trpm8_uniprot else -1)
    for figure, shift in (("fig1", fig1_shift), ("fig2", fig1_shift + 4)):
        row[f"{figure}_residue"] = f"{sequence[ring[0]].get(lower_gate[ring[0]] + shift)}{lower_gate[ring[0]] + shift}"
        ca_distances = []
        closest_distances = []
        for chain_1, chain_2 in diagonal_pairs:
            atoms_1 = residues.get((chain_1, lower_gate[chain_1] + shift), {})
            atoms_2 = residues.get((chain_2, lower_gate[chain_2] + shift), {})
            if "CA" in atoms_1 and "CA" in atoms_2:
                heavy_1 = np.array([vectors[0] for vectors in atoms_1.values()])
                heavy_2 = np.array([vectors[0] for vectors in atoms_2.values()])
                ca_distances.append(np.linalg.norm(atoms_1["CA"][0] - atoms_2["CA"][0]))
                closest_distances.append(np.linalg.norm(heavy_1[:, None] - heavy_2[None], axis=2).min())
        if not ca_distances:
            row[f"{figure}_included"] = False
            continue
        row[f"{figure}_CA_CA_A"] = round(float(np.mean(ca_distances)), 2)
        row[f"{figure}_closest_A"] = round(float(np.mean(closest_distances)), 2)

        # does the side chain face the pore (Gly or missing side chains count as facing)
        cacb_angles = []
        inward_reaches = []
        for chain_id in chains:
            residue_atoms = residues.get((chain_id, lower_gate[chain_id] + shift), {})
            side_chain = {name: vectors[1] for name, vectors in residue_atoms.items() if name not in backbone_atoms}
            if "CA" not in residue_atoms or not side_chain:
                cacb_angles = []
                inward_reaches = []
                break
            ca_radial = residue_atoms["CA"][1]
            inward_reaches.append(np.linalg.norm(ca_radial) - min(np.linalg.norm(vector) for vector in side_chain.values()))
            if "CB" in side_chain:
                ca_to_cb = side_chain["CB"] - ca_radial
                cosine_value = ca_to_cb @ -ca_radial / (np.linalg.norm(ca_to_cb) * np.linalg.norm(ca_radial))
                cacb_angles.append(np.degrees(np.arccos(np.clip(cosine_value, -1, 1))))
        cacb_angle = round(float(np.mean(cacb_angles)), 1) if cacb_angles else None
        inward_reach = round(float(np.mean(inward_reaches)), 2) if inward_reaches else None
        row[f"{figure}_CA_CB_angle"] = cacb_angle
        row[f"{figure}_inward_reach_A"] = inward_reach
        row[f"{figure}_included"] = not ((cacb_angle is not None and cacb_angle > 90) or (inward_reach is not None and inward_reach < 1.0))

Manual calls after visual inspection: Fig 1 drops 8JKM and keeps 8J1H, Fig 2 drops 6MJ2 and keeps 6PUS and 9B3V.

In [ ]:
manual_calls = {("fig1", "8JKM"): False, ("fig1", "8J1H"): True,
                ("fig2", "6MJ2"): False, ("fig2", "6PUS"): True, ("fig2", "9B3V"): True}
for row in rows:
    for (figure, pdb_id), keep in manual_calls.items():
        if row["pdb_id"] == pdb_id:
            row[f"{figure}_included"] = keep

print(Counter(row["excluded"] or "kept" for row in rows))
for figure in ("fig1", "fig2"):
    used = [row for row in rows if row.get(f"{figure}_included") and f"{figure}_CA_CA_A" in row]
    print(figure, Counter(row["S6"] for row in used))

csv_columns = ["pdb_id", "protein", "organism", "uniprot", "S6", "excluded"]
for figure in ("fig1", "fig2"):
    csv_columns += [f"{figure}_residue", f"{figure}_CA_CA_A", f"{figure}_closest_A", f"{figure}_CA_CB_angle", f"{figure}_inward_reach_A", f"{figure}_included"]
with open("lower_gate_survey.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_columns, restval="")
    writer.writeheader()
    writer.writerows(rows)